# Optical Character Recognition (OCR)

> **Warning**
>
> For the following tutorial, we use [DINUM](https://www.numerique.gouv.fr/) instance of OpenGateLLM, called [Albert API](https://albert.api.etalab.gouv.fr/swagger). If your are not a user of this instance, please refer to the [OpenGateLLM readme](https://github.com/etalab-ia/OpenGateLLM?tab=readme-ov-file#-tutorials--guides) to install and configure your own instance. You need to have a image-to-text model to run this tutorial.


OpenGateLLM provides an OCR endpoint based on the Mistral API convention, `/v1/ocr`. For more information on the Mistral API convention, please refer to their [documentation](https://docs.mistral.ai/api/endpoint/ocr).

This tutorial demonstrates how to:
1. Load an image (receipt) and a PDF from Hugging Face datasets
2. Display the documents
3. Perform OCR using Albert API on both documents

In [ ]:
%pip install -qU mistralai requests datasets pillow

import base64
import io
import os

from datasets import load_dataset
from mistralai import Mistral
from PIL import Image
import requests

/Users/leoguillaume/Code/etalab/OpenGateLLM/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


First, setup your API key and server URL. You can set the `ALBERT_API_KEY` environment variable or provide it directly, then initialize the Mistral client.

In [ ]:
# Configuration
server_url = "https://albert.api.etalab.gouv.fr"
api_key = os.getenv("ALBERT_API_KEY")

# Initialize Mistral client
client = Mistral(server_url=server_url, api_key=api_key)

# Headers for direct requests
headers = {"Authorization": f"Bearer {api_key}"}

NameError: name 'os' is not defined

In [ ]:
def encode_image_data(pil_image):
    """Convert PIL image to base64 encoded JPEG string."""
    # Convert to RGB if RGBA / transparency
    if pil_image.mode in ("RGBA", "LA"):
        pil_image = pil_image.convert("RGB")
    buf = io.BytesIO()
    pil_image.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

## Get Available OCR Models

Retrieve the list of available models with `/v1/models` endpoint to perform OCR. These models have the type `image-to-text`.

In [ ]:
# Get available models
models = client.models.list().data

# Find image-to-text models
ocr_models = [model for model in models if model.type == "image-to-text"]

if ocr_models:
    model = ocr_models[0]["id"]
    print(f"OCR model found: {model}")
else:
    print("No image-to-text models found. Please check your configuration.")
    model = None

Chat model found: albert-large


## Load Documents from Hugging Face

We'll load an image (receipt) and a PDF from Hugging Face datasets.

In [ ]:
# Load a receipt image from Hugging Face OCR benchmark dataset
print("Loading receipt image from Hugging Face...")
receipt_dataset = load_dataset("getomni-ai/ocr-benchmark", split="test")
receipt_sample = None
for idx, sample in enumerate(receipt_dataset):
    # Look for a receipt-like image (you can adjust this filter)
    if "receipt" in str(sample.get("metadata", "")).lower() or idx < 5:
        receipt_sample = sample
        break

if receipt_sample:
    receipt_image = receipt_sample["image"]
    print(f"Receipt image loaded: {receipt_image.size}")
else:
    # Fallback: take first sample
    receipt_sample = receipt_dataset[0]
    receipt_image = receipt_sample["image"]
    print(f"Image loaded: {receipt_image.size}")

In [ ]:
# Load a PDF from Hugging Face
# Using a dataset that contains PDFs or document URLs
print("Loading PDF from Hugging Face...")
try:
    # Try to load a dataset with PDFs
    pdf_dataset = load_dataset("mozilla-foundation/common_voice_13_0", split="train", streaming=True)
    # For this example, we'll use a sample PDF URL
    # In practice, you would extract PDF URLs from the dataset
    pdf_url = "https://upload.wikimedia.org/wikipedia/commons/1/12/Guide_de_la_syntaxe_Wiki.pdf"
    print(f"PDF URL: {pdf_url}")
except Exception as e:
    print(f"Could not load PDF dataset: {e}")
    # Fallback PDF URL
    pdf_url = "https://upload.wikimedia.org/wikipedia/commons/1/12/Guide_de_la_syntaxe_Wiki.pdf"
    print(f"Using fallback PDF URL: {pdf_url}")

## Display Documents

Let's display the image and show information about the PDF.

In [ ]:
# Display the receipt image
print("Receipt Image:")
display(receipt_image)
print(f"\nImage size: {receipt_image.size}")
print(f"Image mode: {receipt_image.mode}")

In [ ]:
# Display PDF information
print(f"PDF URL: {pdf_url}")
print("\nNote: PDFs will be processed page by page during OCR.")

## Perform OCR with Albert API

Now we'll perform OCR on both the image and the PDF using Albert API.

### OCR on Image (Receipt)

In [ ]:
# Encode the image to base64
base64_image = encode_image_data(receipt_image)
image_url = f"data:image/jpeg;base64,{base64_image}"

# Perform OCR using requests
print("Performing OCR on receipt image...")
response = requests.post(
    url=f"{server_url}/v1/ocr",
    headers=headers,
    json={
        "model": model,
        "document": {
            "type": "image_url",
            "image_url": image_url
        }
    }
)

if response.status_code == 200:
    ocr_result_image = response.json()
    print("OCR completed successfully!")
    print(f"\nNumber of pages: {len(ocr_result_image.get('pages', []))}")
    if ocr_result_image.get('pages'):
        print(f"\nExtracted text (first page):")
        print(ocr_result_image['pages'][0].get('markdown', '')[:500])
        if len(ocr_result_image['pages'][0].get('markdown', '')) > 500:
            print("...")
else:
    print(f"Error: {response.status_code}")
    print(response.text)
    ocr_result_image = None

In [ ]:
# Alternative: Use Mistral client
print("\nUsing Mistral client for OCR...")
try:
    response_client = client.ocr.process(
        model=model,
        document={
            "type": "image_url",
            "image_url": image_url
        },
        include_image_base64=False
    )
    print("OCR completed successfully!")
    print(f"\nNumber of pages: {len(response_client.pages)}")
    if response_client.pages:
        print(f"\nExtracted text (first page):")
        print(response_client.pages[0].markdown[:500] if response_client.pages[0].markdown else "")
        if response_client.pages[0].markdown and len(response_client.pages[0].markdown) > 500:
            print("...")
        print(f"\nUsage info: {response_client.usage_info}")
except Exception as e:
    print(f"Error using Mistral client: {e}")

### OCR on PDF Document

In [ ]:
# Perform OCR on PDF using document_url
print("Performing OCR on PDF document...")
response = requests.post(
    url=f"{server_url}/v1/ocr",
    headers=headers,
    json={
        "model": model,
        "document": {
            "type": "document_url",
            "document_url": pdf_url,
            "document_name": "guide_syntaxe_wiki.pdf"
        }
    }
)

if response.status_code == 200:
    ocr_result_pdf = response.json()
    print("OCR completed successfully!")
    print(f"\nNumber of pages: {len(ocr_result_pdf.get('pages', []))}")
    print(f"Usage info: {ocr_result_pdf.get('usage_info', {})}")
    
    # Display first page
    if ocr_result_pdf.get('pages'):
        print(f"\nExtracted text (first page, first 500 chars):")
        print(ocr_result_pdf['pages'][0].get('markdown', '')[:500])
        if len(ocr_result_pdf['pages'][0].get('markdown', '')) > 500:
            print("...")
        
        # Display all pages summary
        print(f"\n\nSummary of all pages:")
        for page in ocr_result_pdf.get('pages', []):
            page_text = page.get('markdown', '')
            print(f"Page {page.get('index', '?')}: {len(page_text)} characters")
else:
    print(f"Error: {response.status_code}")
    print(response.text)
    ocr_result_pdf = None

In [ ]:
# Alternative: Use Mistral client for PDF
print("\nUsing Mistral client for PDF OCR...")
try:
    response_client_pdf = client.ocr.process(
        model=model,
        document={
            "type": "document_url",
            "document_url": pdf_url,
            "document_name": "guide_syntaxe_wiki.pdf"
        },
        include_image_base64=False
    )
    print("OCR completed successfully!")
    print(f"\nNumber of pages: {len(response_client_pdf.pages)}")
    print(f"Usage info: {response_client_pdf.usage_info}")
    
    if response_client_pdf.pages:
        print(f"\nExtracted text (first page, first 500 chars):")
        print(response_client_pdf.pages[0].markdown[:500] if response_client_pdf.pages[0].markdown else "")
        if response_client_pdf.pages[0].markdown and len(response_client_pdf.pages[0].markdown) > 500:
            print("...")
except Exception as e:
    print(f"Error using Mistral client: {e}")

## Summary

This tutorial demonstrated how to:
1. Load an image (receipt) and a PDF from Hugging Face datasets
2. Display the documents
3. Perform OCR using Albert API on both image and PDF documents

The OCR results include:
- Extracted text in Markdown format
- Page dimensions and metadata
- Usage information (pages processed, document size)

You can now use these OCR results for further processing, such as text analysis, information extraction, or document understanding tasks.